# FAKTA — End-to-End Evaluation
Test the full pipeline: preprocessing → claim extraction → LSTM → evidence → LLM judge → fusion.

In [ ]:
import sys
sys.path.insert(0, 'src')

from preprocessing.cleaning import clean_text
from preprocessing.slang_normalizer import normalize_slang
from preprocessing.feature_extractor import extract_features
from fusion.confidence_fusion import fuse_claim_verdict, compute_evidence_quality
from fusion.aggregation import aggregate_article_verdicts, ClaimVerdict

In [ ]:
# Test preprocessing pipeline
test_texts = [
    "VIRAL!!! Matcha menyebabkan gagal ginjal dan sudah banyak korban meninggal!! Sebarkan sebelum dihapus!!",
    "BMKG mencatat gempa magnitudo 5.2 di Maluku pada tanggal 15 Januari 2025.",
    "Kabar beredar bahwa harga BBM akan naik bulan depan. Pemerintah belum memberikan konfirmasi resmi.",
]

for text in test_texts:
    cleaned = clean_text(text)
    normalized = normalize_slang(cleaned)
    features = extract_features(normalized)
    
    print(f"\nText: {text[:60]}...")
    print(f"  Cleaned: {normalized[:60]}...")
    print(f"  Hoax Score: {features.hoax_score():.3f}")
    print(f"  Caps Ratio: {features.caps_ratio:.3f}")
    print(f"  Provocative Words: {features.provocative_word_count}")
    print(f"  Exclamation Count: {features.exclamation_count}")

In [ ]:
# Test Fusion Engine with all scenarios
print("=" * 70)
print("FUSION ENGINE TEST")
print("=" * 70)

scenarios = [
    {
        "name": "Strong evidence, claim refuted (HOAX)",
        "lstm_hoax": 0.74,
        "llm_verdict": "Refuted",
        "llm_confidence": 0.88,
        "evidence_quality": 0.89,
        "linguistic_hoax": 0.20,
    },
    {
        "name": "No evidence found (NEUTRAL/NEI)",
        "lstm_hoax": 0.76,
        "llm_verdict": "NotEnoughEvidence",
        "llm_confidence": 0.60,
        "evidence_quality": 0.0,
        "linguistic_hoax": 0.15,
        "nei_reason": "no_search_results",
    },
    {
        "name": "Strong evidence SUPPORTS claim (VALID)",
        "lstm_hoax": 0.85,
        "llm_verdict": "Supported",
        "llm_confidence": 0.90,
        "evidence_quality": 0.85,
        "linguistic_hoax": 0.25,
    },
    {
        "name": "Weak evidence, LSTM suspicious",
        "lstm_hoax": 0.80,
        "llm_verdict": "NotEnoughEvidence",
        "llm_confidence": 0.40,
        "evidence_quality": 0.30,
        "linguistic_hoax": 0.35,
        "nei_reason": "ambiguous",
    },
    {
        "name": "Conflicting evidence",
        "lstm_hoax": 0.60,
        "llm_verdict": "Refuted",
        "llm_confidence": 0.70,
        "evidence_quality": 0.70,
        "linguistic_hoax": 0.15,
        "evidence_conflict": 0.6,
    },
]

for s in scenarios:
    params = {k: v for k, v in s.items() if k != 'name'}
    result = fuse_claim_verdict(**params)
    
    print(f"\n{ s['name'] }:")
    print(f"  Final Hoax: {result.final_hoax_score:.4f}")
    print(f"  Confidence: {result.confidence:.4f}")
    print(f"  Verdict:    {result.verdict}")
    print(f"  Mode:       {result.mode}")
    print(f"  LLM Signal: {result.llm_signal_raw:.4f}")

In [ ]:
# Test Article Aggregation
print("=" * 70)
print("ARTICLE AGGREGATION TEST")
print("=" * 70)

# Simulate article with mixed claims
claims_data = [
    {"text": "Matcha menyebabkan gagal ginjal", "type": "causal", "importance": 1.0, "lstm": 0.74, "llm_v": "Refuted", "llm_c": 0.88, "eq": 0.89},
    {"text": "BPOM sudah meneliti matcha", "type": "factual", "importance": 0.7, "lstm": 0.40, "llm_v": "Supported", "llm_c": 0.92, "eq": 0.85},
    {"text": "Matcha enak diminum panas", "type": "opinion", "importance": 0.3, "lstm": 0.20, "llm_v": "NotEnoughEvidence", "llm_c": 0.5, "eq": 0.0},
]

verdicts = []
for c in claims_data:
    r = fuse_claim_verdict(
        lstm_hoax=c['lstm'], llm_verdict=c['llm_v'], llm_confidence=c['llm_c'],
        evidence_quality=c['eq'], linguistic_hoax=0.1,
    )
    verdicts.append(ClaimVerdict(
        claim_text=c['text'], claim_type=c['type'], importance=c['importance'],
        verdict=r.verdict, final_hoax_score=r.final_hoax_score,
        confidence=r.confidence, mode=r.mode, evidence_sources=['Test'], reasoning='',
    ))

result = aggregate_article_verdicts(verdicts)
print(f"Article Verdict: {result['verdict']}")
print(f"Confidence: {result['confidence']}")
print(f"Summary: {result['summary']}")
print(f"Stats: {result['claim_stats']}")

In [ ]:
# Test with API (requires GEMINI_API_KEY)
import os

if os.environ.get('GEMINI_API_KEY'):
    print("GEMINI_API_KEY found, running full pipeline test...")
    
    from claim_extraction.gemini_extractor import GeminiClaimExtractor
    from judge.gemini_evidence_judge import GeminiEvidenceJudge
    
    # Claim extraction
    extractor = GeminiClaimExtractor()
    text = "Viral! Matcha menyebabkan gagal ginjal dan sudah banyak korban meninggal."
    claims = extractor.extract_claims(text)
    
    print(f"\nExtracted {len(claims)} claims:")
    for c in claims:
        print(f"  [{c.claim_type}] {c.claim_text} (importance: {c.importance})")
    
    # Evidence judge
    judge = GeminiEvidenceJudge()
    evidence = [
        {
            "source": "BPOM",
            "text": "BPOM menyatakan matcha aman dikonsumsi. Tidak ada bukti matcha menyebabkan gagal ginjal.",
            "url": "https://bpom.go.id/"
        }
    ]
    
    result = judge.judge("Matcha menyebabkan gagal ginjal", evidence)
    print(f"\nJudge verdict: {result.llm_verdict}")
    print(f"Confidence: {result.llm_confidence}")
    print(f"Reasoning: {result.reasoning}")
else:
    print("GEMINI_API_KEY not set. Skipping LLM tests.")
    print("Set the env variable and re-run to test claim extraction and evidence judge.")